# Build RAG Index on Kaggle GPU

1. Upload `records.jsonl` as Kaggle dataset `guap-raw`
2. Enable **GPU T4**
3. **Run All**
4. Download `rag_index.tar.gz` from Output

In [ ]:
!pip install -q transformers==4.51.0 sentence-transformers==5.1.1 einops
!pip install -q faiss-cpu bs4 pypdf python-docx pymupdf pytesseract flagembedding

### ⚠️ Restart runtime after install
Go to **Runtime → Restart session**, then run all cells below (skip the cell above).

In [ ]:
import transformers
print(f'transformers=={transformers.__version__}')
assert transformers.__version__ == '4.51.0', f'Expected 4.51.0, got {transformers.__version__}. Restart runtime!'

In [ ]:
!rm -rf /tmp/llm-speaker-core
!GIT_LFS_SKIP_SMUDGE=1 GIT_TERMINAL_PROMPT=0 git clone https://github.com/chudinovAI/llm-speaker-core.git /tmp/llm-speaker-core

import sys
sys.path.insert(0, '/tmp/llm-speaker-core/src')
print('OK')

In [ ]:
from pathlib import Path

RAW_CANDIDATES = [
    Path('/kaggle/input/guap-raw/records.jsonl'),
    Path('/tmp/llm-speaker-core/data/raw/cloudflare/latest/records.jsonl'),
]
RAW_RECORDS = next((p for p in RAW_CANDIDATES if p.exists()), None)
assert RAW_RECORDS is not None, f'records.jsonl not found in {RAW_CANDIDATES}'
print(f'Using: {RAW_RECORDS}')

OUT_DIR = Path('/kaggle/working/rag_output')
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / 'indexes/bm25').mkdir(parents=True, exist_ok=True)
(OUT_DIR / 'indexes/faiss').mkdir(parents=True, exist_ok=True)
(OUT_DIR / 'normalized').mkdir(parents=True, exist_ok=True)

## Step 1: Normalize + Chunk + BM25 (без dense)

In [ ]:
import json
from llm_speaker_core.ingest.normalize import (
    load_cloudflare_documents, dedupe_documents, build_chunk_corpus,
    write_documents, write_chunks,
)
from llm_speaker_core.retrieval.lexical import LexicalIndex
from llm_speaker_core.retrieval.schemas import IndexManifest
from dataclasses import asdict

documents = dedupe_documents(load_cloudflare_documents(RAW_RECORDS))
chunks = build_chunk_corpus(documents)
write_documents(OUT_DIR / 'normalized/documents.jsonl', documents)
write_chunks(OUT_DIR / 'normalized/chunks.jsonl', chunks)
print(f'Documents: {len(documents)}, Chunks: {len(chunks)}')

lexical = LexicalIndex.build(chunks)
lexical.save(OUT_DIR / 'indexes/bm25/index.json')
print('BM25 index built')

## Step 2: Encode chunks with Giga-Embeddings-instruct (GPU)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = 'ai-sage/Giga-Embeddings-instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
model.eval()
model.cuda()
print(f'Model loaded on {next(model.parameters()).device}')

In [ ]:
import numpy as np

def encode_texts(texts, batch_size=8):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(batch, padding=True, truncation=True, max_length=4096, return_tensors='pt')
        encoded = {k: v.cuda() for k, v in encoded.items()}
        with torch.no_grad():
            out = model(**encoded, return_embeddings=True)
        if isinstance(out, torch.Tensor):
            emb = out
        elif hasattr(out, 'last_hidden_state'):
            emb = out.last_hidden_state[:, 0]
        else:
            emb = out[0][:, 0]
        all_embeddings.append(emb.cpu().float().numpy())
        if (i // batch_size) % 10 == 0:
            print(f'  {i}/{len(texts)}', flush=True)
    result = np.vstack(all_embeddings)
    norms = np.linalg.norm(result, axis=1, keepdims=True)
    norms[norms == 0] = 1
    return result / norms

chunk_texts = [c.text for c in chunks]
print(f'Encoding {len(chunk_texts)} chunks...')
vectors = encode_texts(chunk_texts)
print(f'Done: {vectors.shape}')

## Step 3: Save dense index + manifest

In [ ]:
import faiss
import datetime

dense_json_path = OUT_DIR / 'indexes/faiss/index.json'
vectors_path = OUT_DIR / 'indexes/faiss/index.vectors.npy'
faiss_path = OUT_DIR / 'indexes/faiss/index.faiss'

np.save(vectors_path, vectors.astype(np.float32))
dense_json_path.write_text(json.dumps({
    'model_name': MODEL_NAME,
    'available': True,
    'chunks': [c.__dict__ for c in chunks],
}, ensure_ascii=False), encoding='utf-8')

index = faiss.IndexFlatIP(vectors.shape[1])
index.add(vectors.astype(np.float32))
faiss.write_index(index, str(faiss_path))

manifest = IndexManifest(
    version='hybrid-rag-v3',
    corpus_checksum=str(abs(hash(''.join(c.content_hash for c in chunks))) % (10**16)),
    lexical_path='indexes/bm25/index.json',
    dense_path='indexes/faiss/index.json',
    reranker_model='BAAI/bge-reranker-v2-m3',
    embedding_model=MODEL_NAME,
    built_at=datetime.datetime.utcnow().isoformat() + 'Z',
    doc_count=len({c.doc_id for c in chunks}),
    chunk_count=len(chunks),
    metadata={'storage': 'faiss+jsonl', 'dense_available': True, 'reranker_available': True},
)
(OUT_DIR / 'index_manifest.json').write_text(
    json.dumps(asdict(manifest), ensure_ascii=False, indent=2), encoding='utf-8'
)
print('Dense index + manifest saved')
print(json.dumps(asdict(manifest), indent=2, ensure_ascii=False))

## Step 4: Pack archive for download

In [ ]:
import tarfile

archive_path = '/kaggle/working/rag_index.tar.gz'
with tarfile.open(archive_path, 'w:gz') as tar:
    tar.add(OUT_DIR / 'index_manifest.json', arcname='data/index_manifest.json')
    tar.add(OUT_DIR / 'indexes/bm25/index.json', arcname='data/indexes/bm25/index.json')
    tar.add(OUT_DIR / 'indexes/faiss/index.json', arcname='data/indexes/faiss/index.json')
    tar.add(vectors_path, arcname='data/indexes/faiss/index.vectors.npy')
    tar.add(faiss_path, arcname='data/indexes/faiss/index.faiss')
    tar.add(OUT_DIR / 'normalized/documents.jsonl', arcname='data/normalized/documents.jsonl')
    tar.add(OUT_DIR / 'normalized/chunks.jsonl', arcname='data/normalized/chunks.jsonl')

size_mb = Path(archive_path).stat().st_size / 1024 / 1024
print(f'Archive: {archive_path} ({size_mb:.1f} MB)')
print('Download and run: cd llm-speaker-core && tar xzf ~/Downloads/rag_index.tar.gz')